
# H17: Initial Gradient Anisotropy as a Toy Predictor of Muon Advantage

**Counterpart script:** `run_experiment.py` in the same experiment directory.

This notebook is the presentation and analysis wrapper for the script implementation. It **does not duplicate the experiment core**: it imports the script, calls its `run_experiment()` function, and inspects the structured results object that the script returns.

## Study scope and claim calibration

This experiment asks a narrow predictor-style question on a toy problem:

> Across five small synthetic architectures, is larger **initial first-layer gradient anisotropy** associated with larger **Muon-over-SGD benefit** after short learning-rate sweeps?

Important limitations:
- this is a **five-architecture synthetic regression study**;
- the learning rates are chosen from **predefined candidate grids**, not continuous optimization;
- the architecture-level correlation is **descriptive**, not a formal significance claim;
- the experiment does **not** by itself prove the mechanism behind Muon's behavior.



## Notebook structure

This notebook is organized as follows:
1. locate and import the script implementation safely;
2. log reproducibility and runtime settings;
3. run either a quick smoke configuration or the full default toy study;
4. present anisotropy summaries, LR sweep summaries, final loss / advantage summaries, and the anisotropy-vs-advantage scatter;
5. report the heuristic T1/T2/T3 checks and a calibrated conclusion.

The full default study can be slow on CPU, especially for the `wide` architecture during the Muon learning-rate sweep. The notebook therefore exposes a **safe smoke mode** by default while making the exact full-study settings explicit.


In [ ]:
from pathlib import Path
import importlib.util
import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except ImportError:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text

try:
    import pandas as pd
    HAVE_PANDAS = True
except ImportError:
    pd = None
    HAVE_PANDAS = False

EXPERIMENT_RELATIVE = Path(
    "experiments/Tier_4_Falsification_Stress_Tests/H17_ANISOTROPY_PREDICTS_BENEFIT/run_experiment.py"
)

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / EXPERIMENT_RELATIVE).exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root from the current working directory. "
        "Please start the notebook from somewhere inside the Muon_as_RG_Gauge_Fixing repo."
    )

REPO_ROOT = find_repo_root(Path.cwd())
SCRIPT_PATH = REPO_ROOT / EXPERIMENT_RELATIVE
EXPERIMENT_DIR = SCRIPT_PATH.parent

spec = importlib.util.spec_from_file_location("h17_run_experiment", SCRIPT_PATH)
h17 = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(h17)

plt.style.use("seaborn-v0_8-whitegrid")

print(f"Repository root: {REPO_ROOT}")
print(f"Experiment directory: {EXPERIMENT_DIR}")
print(f"Imported script: {SCRIPT_PATH}")
print(f"Pandas available: {HAVE_PANDAS}")



## Reproducibility, runtime, and execution mode

The script exposes the exact default toy-study configuration. This notebook can either:
- run that full default study, or
- run a reduced smoke configuration for validation and quick inspection.

The smoke configuration is the notebook default because it is safer for repeated execution. Set `RUN_FULL_EXPERIMENT = True` below to reproduce the script's full default study.


In [ ]:

FULL_CONFIG = h17.get_default_config()
RUN_FULL_EXPERIMENT = False

SMOKE_OVERRIDES = {
    "arch_names": ["deep_linear", "relu_mlp"],
    "num_steps": 20,
    "num_seeds": 2,
    "seeds": h17.get_default_seeds(2),
    "lr_sweep_num_seeds": 2,
    "bootstrap_samples": 200,
    "make_plot": False,
    "save_plot": False,
    "show_plot": False,
    "verbose": False,
}

FULL_RUN_OVERRIDES = {
    "make_plot": False,
    "save_plot": False,
    "show_plot": False,
    "verbose": True,
}

RUN_KWARGS = FULL_RUN_OVERRIDES if RUN_FULL_EXPERIMENT else SMOKE_OVERRIDES
ACTIVE_ARCH_NAMES = RUN_KWARGS.get("arch_names", FULL_CONFIG["arch_names"])
ACTIVE_NUM_SEEDS = RUN_KWARGS.get("num_seeds", FULL_CONFIG["num_seeds"])
ACTIVE_LR_SWEEP_NUM_SEEDS = RUN_KWARGS.get("lr_sweep_num_seeds", FULL_CONFIG["lr_sweep_num_seeds"])
ACTIVE_NUM_STEPS = RUN_KWARGS.get("num_steps", FULL_CONFIG["num_steps"])
ACTIVE_SEEDS = RUN_KWARGS.get("seeds", FULL_CONFIG["seeds"])

def estimate_training_budget(arch_names, num_seeds, lr_sweep_num_seeds, n_sgd, n_muon):
    lr_sweep_runs = len(arch_names) * (n_sgd + n_muon) * lr_sweep_num_seeds
    final_eval_runs = len(arch_names) * 2 * num_seeds
    return {
        "lr_sweep_runs": lr_sweep_runs,
        "final_eval_runs": final_eval_runs,
        "total_train_runs": lr_sweep_runs + final_eval_runs,
    }

full_budget = estimate_training_budget(
    FULL_CONFIG["arch_names"],
    FULL_CONFIG["num_seeds"],
    FULL_CONFIG["lr_sweep_num_seeds"],
    len(FULL_CONFIG["lr_sgd_candidates"]),
    len(FULL_CONFIG["lr_muon_candidates"]),
)

active_budget = estimate_training_budget(
    ACTIVE_ARCH_NAMES,
    ACTIVE_NUM_SEEDS,
    ACTIVE_LR_SWEEP_NUM_SEEDS,
    len(FULL_CONFIG["lr_sgd_candidates"]),
    len(FULL_CONFIG["lr_muon_candidates"]),
)

mode = "FULL DEFAULT STUDY" if RUN_FULL_EXPERIMENT else "SMOKE / VALIDATION MODE"
print(f"Execution mode: {mode}")
print()
print("Full default configuration from the script:")
for key in [
    "arch_names",
    "num_steps",
    "num_seeds",
    "batch_size",
    "momentum",
    "ns_iters",
    "lr_sweep_num_seeds",
    "lr_sgd_candidates",
    "lr_muon_candidates",
    "seeds",
]:
    print(f"  {key}: {FULL_CONFIG[key]}")
print()
print("Estimated training budget for full defaults:", full_budget)
print("Estimated training budget for current notebook run:", active_budget)
print(f"Active steps: {ACTIVE_NUM_STEPS}")
print(f"Active seeds: {ACTIVE_SEEDS}")



## Metric definitions used by the script

The script computes the following quantities.

### Initial anisotropy
For each architecture and seed:
- generate Gaussian synthetic regression data;
- initialize the network weights;
- compute the first-layer gradient at initialization;
- take its singular values and define anisotropy as
  \(\sigma_{\max} / \sigma_{\min}\), with a floor on \(\sigma_{\min}\) for numerical safety.

The notebook reports the seed-level values, architecture-level mean and standard deviation, and also retains `sigma_max`, `sigma_min`, and effective rank in the raw results.

### Learning-rate selection
For each architecture and optimizer (`sgd`, `muon`):
- evaluate each candidate LR on a subset of seeds;
- compute the mean over **finite final losses only**;
- pick the candidate with the smallest mean finite loss.

This is intentionally reported as **best LR among candidates**, not a global optimum.

### Muon advantage
After picking the best candidate LR for each optimizer, the script evaluates all seeds and defines

\[
	ext{Muon advantage} = rac{	ext{mean SGD loss}}{	ext{mean Muon loss}}.
\]

This is a **ratio of means**, not a mean of per-seed ratios.


In [ ]:

def rows_to_frame(rows, sort_by=None, ascending=False):
    if HAVE_PANDAS:
        frame = pd.DataFrame(rows)
        if sort_by is not None and len(frame) > 0:
            frame = frame.sort_values(sort_by, ascending=ascending)
        return frame.reset_index(drop=True)
    if sort_by is not None:
        rows = sorted(rows, key=lambda row: row[sort_by], reverse=not ascending)
    return rows

def display_rows(rows, sort_by=None, ascending=False, columns=None):
    frame = rows_to_frame(rows, sort_by=sort_by, ascending=ascending)
    if HAVE_PANDAS:
        if columns is not None:
            display(frame.loc[:, columns])
        else:
            display(frame)
    else:
        if columns is not None:
            trimmed = [{k: row[k] for k in columns} for row in frame]
            print(trimmed)
        else:
            print(frame)
    return frame

def architecture_summary_rows(results):
    rows = []
    for arch in results["config"]["arch_names"]:
        summary = results["architecture_results"][arch]["summary"].copy()
        summary["dims"] = str(summary["dims"])
        rows.append(summary)
    return rows

def anisotropy_seed_rows(results):
    rows = []
    for arch in results["config"]["arch_names"]:
        arch_result = results["architecture_results"][arch]
        for record in arch_result["anisotropy"]["per_seed"]:
            rows.append({
                "architecture": arch,
                "seed": record["seed"],
                "anisotropy": record["anisotropy"],
                "sigma_max": record["sigma_max"],
                "sigma_min": record["sigma_min"],
                "effective_rank": record["effective_rank"],
            })
    return rows

def selected_lr_rows(results):
    rows = []
    for arch in results["config"]["arch_names"]:
        arch_result = results["architecture_results"][arch]
        for optimizer in ["sgd", "muon"]:
            sweep = arch_result["lr_sweeps"][optimizer]
            total = len(sweep["seed_subset"])
            best_row = None
            for row in sweep["results"]:
                if abs(row["lr"] - sweep["best_lr"]) < 1e-15:
                    best_row = row
                    break
            rows.append({
                "architecture": arch,
                "optimizer": optimizer,
                "best_lr": sweep["best_lr"],
                "best_mean_finite_loss": sweep["best_mean_finite_loss"],
                "finite_count_at_best_lr": best_row["finite_count"] if best_row is not None else None,
                "diverged_count_at_best_lr": best_row["diverged_count"] if best_row is not None else None,
                "lr_sweep_seed_count": total,
            })
    return rows

def lr_sweep_rows(results):
    rows = []
    for arch in results["config"]["arch_names"]:
        arch_result = results["architecture_results"][arch]
        for optimizer in ["sgd", "muon"]:
            sweep = arch_result["lr_sweeps"][optimizer]
            total = len(sweep["seed_subset"])
            for row in sweep["results"]:
                rows.append({
                    "architecture": arch,
                    "optimizer": optimizer,
                    "lr": row["lr"],
                    "mean_finite_loss": row["mean_finite_loss"],
                    "std_finite_loss": row["std_finite_loss"],
                    "finite_count": row["finite_count"],
                    "diverged_count": row["diverged_count"],
                    "finite_fraction": row["finite_count"] / total if total else float("nan"),
                })
    return rows

def final_loss_rows(results):
    rows = []
    for arch in results["config"]["arch_names"]:
        arch_result = results["architecture_results"][arch]
        advantage = arch_result["advantage"]
        for optimizer in ["sgd", "muon"]:
            final_result = arch_result["final_losses"][optimizer]
            rows.append({
                "architecture": arch,
                "optimizer": optimizer,
                "selected_lr": final_result["lr"],
                "mean_loss": final_result["mean"],
                "std_loss": final_result["std"],
                "finite_count": final_result["finite_count"],
                "diverged_count": final_result["diverged_count"],
                "paired_advantage_ratio_of_means": advantage["ratio_of_means"],
            })
    return rows

def advantage_rows(results):
    rows = []
    for arch in results["config"]["arch_names"]:
        arch_result = results["architecture_results"][arch]
        advantage = arch_result["advantage"]
        rows.append({
            "architecture": arch,
            "advantage_ratio_of_means": advantage["ratio_of_means"],
            "advantage_ci_low": advantage["bootstrap_ci"]["low"],
            "advantage_ci_high": advantage["bootstrap_ci"]["high"],
            "paired_finite_count": advantage["paired_finite_count"],
        })
    return rows

def plot_anisotropy_summary(results):
    summaries = architecture_summary_rows(results)
    archs = [row["architecture"] for row in summaries]
    means = np.array([row["anisotropy_mean"] for row in summaries], dtype=float)
    stds = np.array([row["anisotropy_std"] for row in summaries], dtype=float)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.errorbar(archs, means, yerr=stds, fmt="o", capsize=4, linewidth=1.5)
    ax.set_yscale("log")
    ax.set_ylabel("Initial anisotropy mean ± seed std")
    ax.set_title("Architecture-level initial anisotropy summary")
    ax.set_xlabel("Architecture")
    plt.xticks(rotation=20)
    plt.tight_layout()
    return fig, ax

def plot_lr_sweeps(results):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=False)
    for ax, optimizer in zip(axes, ["sgd", "muon"]):
        for arch in results["config"]["arch_names"]:
            sweep = results["architecture_results"][arch]["lr_sweeps"][optimizer]
            rows = sorted(sweep["results"], key=lambda row: row["lr"])
            lrs = np.array([row["lr"] for row in rows], dtype=float)
            losses = np.array([row["mean_finite_loss"] for row in rows], dtype=float)
            ax.plot(lrs, losses, marker="o", label=arch)
            best_lr = sweep["best_lr"]
            best_loss = sweep["best_mean_finite_loss"]
            ax.scatter([best_lr], [best_loss], s=80, edgecolor="black", zorder=4)
        ax.set_xscale("log")
        ax.set_xlabel("Candidate LR")
        ax.set_ylabel("Mean finite final loss on sweep seeds")
        ax.set_title(f"{optimizer.upper()} LR sweep summary")
        ax.legend(fontsize=8)
    plt.tight_layout()
    return fig, axes

def plot_final_losses_and_advantage(results):
    summaries = architecture_summary_rows(results)
    archs = [row["architecture"] for row in summaries]

    sgd_means = np.array([row["sgd_loss_mean"] for row in summaries], dtype=float)
    sgd_stds = np.array([row["sgd_loss_std"] for row in summaries], dtype=float)
    muon_means = np.array([row["muon_loss_mean"] for row in summaries], dtype=float)
    muon_stds = np.array([row["muon_loss_std"] for row in summaries], dtype=float)
    adv_means = np.array([row["advantage_ratio_of_means"] for row in summaries], dtype=float)
    adv_low = np.array([row["advantage_ci_low"] for row in summaries], dtype=float)
    adv_high = np.array([row["advantage_ci_high"] for row in summaries], dtype=float)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    axes[0].errorbar(archs, sgd_means, yerr=sgd_stds, fmt="o-", capsize=4, label="SGD")
    axes[0].errorbar(archs, muon_means, yerr=muon_stds, fmt="o-", capsize=4, label="Muon")
    axes[0].set_yscale("log")
    axes[0].set_title("Final loss by architecture and optimizer")
    axes[0].set_ylabel("Mean final loss ± seed std")
    axes[0].set_xlabel("Architecture")
    axes[0].legend()
    axes[0].tick_params(axis="x", rotation=20)

    lower = np.clip(adv_means - adv_low, 0.0, None)
    upper = np.clip(adv_high - adv_means, 0.0, None)
    axes[1].errorbar(archs, adv_means, yerr=np.vstack([lower, upper]), fmt="o", capsize=4)
    axes[1].axhline(1.0, linestyle="--", color="gray", linewidth=1)
    axes[1].set_title("Muon advantage by architecture")
    axes[1].set_ylabel("Mean SGD loss / mean Muon loss")
    axes[1].set_xlabel("Architecture")
    axes[1].tick_params(axis="x", rotation=20)

    plt.tight_layout()
    return fig, axes



## Run the experiment through the script API

The cell below is the only place where the experiment is executed. All downstream tables and figures consume the returned `results` object.


In [ ]:

results = h17.run_experiment(**RUN_KWARGS)

print(f"Study title: {results['title']}")
print(f"Scope note: {results['scope_note']}")
print(f"Architectures evaluated in this run: {results['config']['arch_names']}")
print(f"Runtime: {results['runtime']['elapsed_sec']:.2f} seconds")
print(f"Linear Spearman r: {results['correlations']['linear']['spearman_r']:.3f}")
print(f"Log10 Spearman r:  {results['correlations']['log10']['spearman_r']:.3f}")
if not RUN_FULL_EXPERIMENT:
    print("NOTE: Current outputs are from smoke mode, so do not interpret them as the full H17 result.")



## Architecture anisotropy summary

The script measures initial anisotropy separately for each seed, then aggregates by architecture. Because the raw condition-number-style quantity can span many orders of magnitude, the figure uses a log y-axis.


In [ ]:

aniso_summary = architecture_summary_rows(results)
aniso_seed_detail = anisotropy_seed_rows(results)

_ = display_rows(
    aniso_summary,
    columns=[
        "architecture",
        "activation",
        "dims",
        "anisotropy_mean",
        "anisotropy_std",
    ],
)

_ = display_rows(
    aniso_seed_detail,
    columns=[
        "architecture",
        "seed",
        "anisotropy",
        "sigma_max",
        "sigma_min",
        "effective_rank",
    ],
)

fig, ax = plot_anisotropy_summary(results)
plt.show()



### Interpretation

This section is descriptive only. The error bars summarize variation across seeds, not formal uncertainty about a population quantity. Also note that \(\sigma_{\max} / \sigma_{\min}\) can be dominated by the smallest singular value, so extremely large anisotropy values should be read cautiously.



## Learning-rate sweeps and selected candidate LRs

The script evaluates every candidate LR for SGD and Muon on a subset of seeds, keeps the **mean over finite final losses**, and selects the best candidate LR for each optimizer / architecture pair.

The tables below therefore expose both performance and stability information (`finite_count`, `diverged_count`) instead of reporting only the chosen LR.


In [ ]:

selected_lr_table = selected_lr_rows(results)
lr_sweep_table = lr_sweep_rows(results)

_ = display_rows(
    selected_lr_table,
    columns=[
        "architecture",
        "optimizer",
        "best_lr",
        "best_mean_finite_loss",
        "finite_count_at_best_lr",
        "diverged_count_at_best_lr",
        "lr_sweep_seed_count",
    ],
)

_ = display_rows(
    lr_sweep_table,
    columns=[
        "architecture",
        "optimizer",
        "lr",
        "mean_finite_loss",
        "std_finite_loss",
        "finite_count",
        "diverged_count",
        "finite_fraction",
    ],
)

fig, axes = plot_lr_sweeps(results)
plt.show()



### Interpretation

These LR sweeps should be interpreted as a bounded grid search, not a continuous optimum finder. Because the selection rule uses mean **finite** loss, the stability counts matter: a candidate LR can look good on the subset of surviving runs even if some seeds diverge.



## Final losses and Muon advantage

After selecting candidate LRs, the script re-evaluates all seeds and computes the architecture-level Muon advantage as

\[
	ext{advantage} = rac{	ext{mean SGD loss}}{	ext{mean Muon loss}}.
\]

The notebook shows both optimizer losses and a bootstrap interval for the ratio-of-means summary.


In [ ]:

final_loss_table = final_loss_rows(results)
adv_table = advantage_rows(results)

_ = display_rows(
    final_loss_table,
    columns=[
        "architecture",
        "optimizer",
        "selected_lr",
        "mean_loss",
        "std_loss",
        "finite_count",
        "diverged_count",
        "paired_advantage_ratio_of_means",
    ],
)

_ = display_rows(
    adv_table,
    columns=[
        "architecture",
        "advantage_ratio_of_means",
        "advantage_ci_low",
        "advantage_ci_high",
        "paired_finite_count",
    ],
)

fig, axes = plot_final_losses_and_advantage(results)
plt.show()



### Interpretation

A Muon advantage above 1 means Muon achieved a lower final loss than SGD under the selected candidate LRs. The bootstrap interval shown here is a descriptive uncertainty summary over finite losses only; it is useful for context but should not be over-read as a definitive inferential statement.



## Anisotropy versus Muon advantage

This is the key architecture-level comparison. The script's plotting helper is used directly so that the notebook and script stay aligned. The fitted trend is descriptive, and the log-scale panel is included because both anisotropy and advantage can vary across orders of magnitude.


In [ ]:

summary_frame = display_rows(
    architecture_summary_rows(results),
    columns=[
        "architecture",
        "anisotropy_mean",
        "anisotropy_std",
        "sgd_lr",
        "muon_lr",
        "sgd_loss_mean",
        "muon_loss_mean",
        "advantage_ratio_of_means",
        "advantage_ci_low",
        "advantage_ci_high",
    ],
)

fig, axes = h17.make_summary_plot(results, save_path=None, show=False)
plt.show()



## Heuristic checks: T1, T2, T3

These checks are kept for continuity with the original experiment framing, but they should be interpreted as heuristic summaries rather than formal hypothesis tests.

- **T1:** is the architecture-level Spearman correlation above 0.5?
- **T2:** does the highest-anisotropy architecture also have the highest Muon advantage?
- **T3:** does the lowest-anisotropy architecture have below-median Muon advantage?


In [ ]:

tests = results["tests"]

heuristic_rows = [
    {
        "test": "T1_spearman_gt_0_5",
        "passed": tests["T1_spearman_gt_0_5"]["passed"],
        "details": f"Spearman r = {tests['T1_spearman_gt_0_5']['spearman_r']:.3f}",
    },
    {
        "test": "T2_top_arch_match",
        "passed": tests["T2_top_arch_match"]["passed"],
        "details": (
            f"top anisotropy = {tests['T2_top_arch_match']['top_anisotropy_architecture']}; "
            f"top advantage = {tests['T2_top_arch_match']['top_advantage_architecture']}"
        ),
    },
    {
        "test": "T3_lowest_anisotropy_below_median_advantage",
        "passed": tests["T3_lowest_anisotropy_below_median_advantage"]["passed"],
        "details": (
            f"lowest anisotropy arch = {tests['T3_lowest_anisotropy_below_median_advantage']['lowest_anisotropy_architecture']}; "
            f"median advantage = {tests['T3_lowest_anisotropy_below_median_advantage']['median_advantage']:.3f}"
        ),
    },
]

_ = display_rows(heuristic_rows, columns=["test", "passed", "details"])

print("Linear correlations:")
print(results["correlations"]["linear"])
print("Log10 correlations:")
print(results["correlations"]["log10"])



## Calibrated conclusion

The notebook should end with the same core interpretation as the script: a bounded, architecture-level descriptive statement rather than a claim of strong significance or mechanistic closure.


In [ ]:
conclusion = results["conclusion"]

mode_note = (
    "These outputs come from the script's full default toy study."
    if RUN_FULL_EXPERIMENT
    else "These outputs come from notebook smoke mode; re-run with RUN_FULL_EXPERIMENT = True for the full default study."
)

conclusion_md = (
    "### Result summary\n"
    f"- **Headline:** {conclusion['headline']}\n"
    f"- **Heuristic tally:** {conclusion['tests_passed']}/3 checks passed.\n"
    f"- **Caveat:** {conclusion['caveat']}\n"
    f"- **Execution note:** {mode_note}\n"
)

display(Markdown(conclusion_md))
